In [1]:
# !WANDB_PROJECT="cfe_finetuning" python3 -m wandb sweep sweep.yaml

In [2]:
import pandas as pd
import json

In [3]:
import pandas as pd
import json
from pathlib import Path
import os

base_dir = Path("finetuning_eval")
json_files = list(base_dir.rglob("metrics.json"))
print(json_files)
all_dfs = []

for json_file in json_files:
    with open(json_file, "r") as f:
        data = json.load(f)

    scores_list = []
    for entry in data.get("all_primary_scores", []):
        alias, score = entry.rsplit(": ", 1)
        scores_list.append({"alias": alias, "primary_score": float(score)})
    df_primary_scores = pd.DataFrame(scores_list)

    tasks_list = []
    for task in data.get("tasks", []):
        entry = {"alias": task["alias"], "num_instances": task.get("num_instances"), "task_suite": json_file.parts[1]}
        entry.update(task.get("metrics", {}))
        tasks_list.append(entry)
    df_tasks = pd.DataFrame(tasks_list)

    df = pd.merge(df_primary_scores, df_tasks, on="alias", how="left")
    df["model"] = os.path.basename(json_file.parent)
    all_dfs.append(df)

df = pd.concat(all_dfs, ignore_index=True)

def rename(name):
    if "arc_easy" in name:
        return "ARC\_E"
    elif "arc_challenge" in name:
        return "ARC\_C"
    elif "boolq" in name:
        return "BoolQ"
    elif "csqa" in name:
        return "CSQA"
    elif "hellaswag" in name:
        return "HSwag"
    elif "mmlu_pro" in name:
        return "MMLU-Pro"
    elif "mmlu" in name:
        return "MMLU"
    elif "openbookqa" in name:
        return "OBQA"
    elif "piqa" in name:
        return "PIQA"
    elif "socialiqa" in name:
        return "SIQA"
    elif "winogrande" in name:
        return "WinoG"
    elif "agi" in name:
        return "AGIEval"
    elif "bbh" in name:
        return "BBH"
    elif "gsm8k" in name:
        return "GSM8K"
    elif "triviaqa" in name:
        return "TriviaQA"
    elif "coqa" in name:
        return "CoQA"
    elif "drop" in name:
        return "DROP"
    elif "jeopardy" in name:
        return "JPRDY"
    elif "naturalqs" in name:
        return "NatQs"
    elif "squad" in name:
        return "SQuAD"
    else:
        return name+" TODO"
df["task"] = df["alias"].apply(rename)
df = df[~df['task'].str.contains("TODO", na=False)]
cols = df.columns[:2].tolist() + sorted(df.columns[2:])
df = df[cols]
df = df[~df["model"].str.contains("lr1e-05")]

[PosixPath('finetuning_eval/mmlu:mc::olmes/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Qwen/Qwen2.5-0.5B-Instruct/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/meta-llama/Llama-3.2-1B-Instruct/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/allenai/OLMo-2-0425-1B-SFT/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42-merged/met

In [4]:
df_f = df[["task_suite", "alias", "primary_score_y", "model", "task"]].copy()
df_f = df_f.groupby(["model", "task"])["primary_score_y"].mean().reset_index()
df_f = df_f.pivot(index="model", columns="task", values="primary_score_y")
df_f.insert(0, "avg", df_f.mean(axis=1, numeric_only=True) 
)
external_models = ["Qwen2.5-0.5B-Instruct", "OLMo-2-0425-1B-SFT", "Llama-3.2-1B-Instruct"]
df_f = df_f.reset_index()
df_f["type"] = df_f["model"].apply(lambda x: "external" if x in external_models else "own")
#





for name, group in df_f.groupby("type"):
    display(group)
    group = group.drop(columns="type").set_index("model")
    for col in group.columns:
        max_val = df_f[col].max()
        def fmt(x):
            formatted = f"{x:.2f}".lstrip("0") if x < 1 else f"{x:.2f}"
            if x == max_val:
                return f"\\textbf{{{formatted}}}"
            return formatted
        
        group[col] = group[col].apply(fmt)


    group.columns = [str(c) for c in group.columns]
    group.columns = [f"\\rotatebox{{45}}{{{str(c)}}}" for c in group.columns]

    group.index = [i.split("_")[0] for i in group.index]
    group.index.name = "LoRA models" if name == "own" else "External models"
    print(group.index)

    file_path = f"tables/olmes-{name}.tex"
    os.makedirs("tables", exist_ok=True)
    with open(file_path, "w") as f:
        f.write(group.to_latex(
            index=True,
            header=name == "own",
            escape=False
        ))

        

task,model,avg,AGIEval,ARC\_C,ARC\_E,BBH,BoolQ,CSQA,CoQA,DROP,...,MMLU,MMLU-Pro,NatQs,OBQA,PIQA,SIQA,SQuAD,TriviaQA,WinoG,type
0,Llama-3.2-1B-Instruct,0.506774,0.349781,0.489761,0.731333,0.371494,0.636000,0.630358,0.669044,0.28838,...,0.464886,0.247208,0.19971,0.538000,0.707333,0.566000,0.789595,0.470840,0.587214,external
2,OLMo-2-0425-1B-SFT,0.525708,0.362203,0.481797,0.747000,0.315006,0.749333,0.645919,0.715931,0.33337,...,0.440424,0.213020,0.17196,0.510667,0.724000,0.581000,0.831776,0.481006,0.597737,external
4,Qwen2.5-0.5B-Instruct,0.444229,0.380186,0.494027,0.713333,0.311631,0.679000,0.599509,0.412742,0.24967,...,0.482305,0.230409,0.08591,0.535333,0.667000,0.561667,0.740223,0.089513,0.529334,external


Index(['Llama-3.2-1B-Instruct', 'OLMo-2-0425-1B-SFT', 'Qwen2.5-0.5B-Instruct'], dtype='object', name='External models')


task,model,avg,AGIEval,ARC\_C,ARC\_E,BBH,BoolQ,CSQA,CoQA,DROP,...,MMLU,MMLU-Pro,NatQs,OBQA,PIQA,SIQA,SQuAD,TriviaQA,WinoG,type
1,Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr...,0.435098,0.240972,0.377418,0.599333,0.319473,0.668000,0.501229,0.651941,0.25152,...,0.295857,0.158291,0.16189,0.392000,0.675000,0.471667,0.725141,0.484636,0.581689,own
3,OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_...,0.515677,0.337964,0.474972,0.742000,0.297515,0.693000,0.597598,0.688414,0.34566,...,0.429064,0.191777,0.18912,0.511333,0.707000,0.557333,0.803783,0.554311,0.610892,own
5,Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr...,0.457176,0.367194,0.489192,0.711667,0.315104,0.660667,0.580126,0.597105,0.23646,...,0.490520,0.225636,0.11735,0.536667,0.661333,0.558667,0.728459,0.191097,0.535649,own


Index(['Llama-3.2-1B', 'OLMo-2-0425-1B', 'Qwen2.5-0.5B'], dtype='object', name='LoRA models')


In [ ]:
# pip install pygsheets

In [ ]:
# pip install lm_eval

In [7]:
!nvidia-smi

Wed Oct  8 11:45:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-16GB           On  |   00000000:06:00.0 Off |                    0 |
| N/A   38C    P0             60W /  300W |    6108MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !python3 ./merge_lora_to_hf.py \
#   --base_model EleutherAI/pythia-31m \
#   --adapter_dir /srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train \
#   --output_dir /srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train-merged


In [ ]:
import multiprocessing as mp
mp.set_start_method('spawn', force=True)


In [10]:
!

In [ ]:
# import subprocess
# import os

# # Change working directory to olmes
# olmes_dir = os.path.join("olmes")
# os.chdir(olmes_dir)

# cmd = [
#     "CUDA_VISIBLE_DEVICES=0 python3", "-m", "oe_eval.launch",
#     "--model","/srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train-merged",
#     "--task", "tulu_3_dev",
#     "--output-dir", "./finetuning_eval/"
  
# ]

# subprocess.run(cmd, check=True)
